In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [12]:
df=pd.read_csv("../data/raw/creditcard.csv")
df=df.drop_duplicates().reset_index(drop=True)
df["Hour"]=(df["Time"]//3600)%24
X=df.drop(columns="Class")
Y=df["Class"]

In [15]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,random_state=2,test_size=0.2,stratify=Y)
print(Y_train.value_counts(normalize=True)*100)
print(Y_test.value_counts(normalize=True)*100)

Class
0    99.833466
1     0.166534
Name: proportion, dtype: float64
Class
0    99.832587
1     0.167413
Name: proportion, dtype: float64


In [17]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, average_precision_score

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, Y_train)
y_pred = dummy.predict(X_test)

print(classification_report(Y_test, y_pred, digits=4, zero_division=0))
print("PR-AUC:", average_precision_score(Y_test, dummy.predict_proba(X_test)[:, 1]))

              precision    recall  f1-score   support

           0     0.9983    1.0000    0.9992     56651
           1     0.0000    0.0000    0.0000        95

    accuracy                         0.9983     56746
   macro avg     0.4992    0.5000    0.4996     56746
weighted avg     0.9967    0.9983    0.9975     56746

PR-AUC: 0.0016741268107003137


In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix,roc_auc_score

scale_cols=["Time","Amount","Hour"]
prep=ColumnTransformer(
    [("scale",RobustScaler(),scale_cols)],
    remainder="passthrough"
)
basline=Pipeline([
    ("prep",prep),
    ("model",LogisticRegression(class_weight="balanced",max_iter=1000 ))

])
basline.fit(X_train,Y_train)
Y_pred=basline.predict(X_test)
Y_proba=basline.predict_proba(X_test)[:,1]
print(classification_report(Y_test, y_pred, digits=4))
print(confusion_matrix(Y_test, y_pred))
print("PR-AUC:", average_precision_score(Y_test, Y_proba))

              precision    recall  f1-score   support

           0     0.9983    1.0000    0.9992     56651
           1     0.0000    0.0000    0.0000        95

    accuracy                         0.9983     56746
   macro avg     0.4992    0.5000    0.4996     56746
weighted avg     0.9967    0.9983    0.9975     56746

[[56651     0]
 [   95     0]]
PR-AUC: 0.6939579065039058


c:\Users\Dell\Desktop\ML_Project\fraudshield\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Dell\Desktop\ML_Project\fraudshield\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Dell\Desktop\ML_Project\fraudshield\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifi

In [23]:
basline.fit(X_train, Y_train)

y_pred_lr = basline.predict(X_test)
y_proba_lr = basline.predict_proba(X_test)[:, 1]

print("Predicted frauds:", y_pred_lr.sum())
print(classification_report(Y_test, y_pred_lr, digits=4))
print(confusion_matrix(Y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(Y_test, y_proba_lr))
print("PR-AUC :", average_precision_score(Y_test, y_proba_lr))

Predicted frauds: 1311
              precision    recall  f1-score   support

           0     0.9998    0.9784    0.9890     56651
           1     0.0656    0.9053    0.1223        95

    accuracy                         0.9783     56746
   macro avg     0.5327    0.9418    0.5557     56746
weighted avg     0.9983    0.9783    0.9875     56746

[[55426  1225]
 [    9    86]]
ROC-AUC: 0.982674528902263
PR-AUC : 0.6939579065039058
